# TP2 - Quasi-Newton Methods and Constrained Optimization

## Lecture Material and Results Used

- **Logistic regression objective**: a smooth convex empirical-risk objective used for binary classification.
- **BFGS update**: approximate inverse Hessian updates use pairs $s_k=x_{k+1}-x_k$ and $y_k=\nabla f(x_{k+1})-\nabla f(x_k)$.
- **Curvature condition**: BFGS needs $s_k^Ty_k>0$ to keep the inverse-Hessian approximation positive definite.
- **Line search**: quasi-Newton steps are usually paired with sufficient-decrease checks.
- **Projected gradient descent**: for constraints $x\in C$, iterate $x_{k+1}=\Pi_C(x_k-\beta\nabla f(x_k))$.
- **Penalty and barrier ideas**: convert constrained problems into unconstrained approximations by penalizing infeasibility or approaching the boundary from inside.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(41)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})


## 1. Logistic Regression Objective

Generate binary labels from a logistic model and fit the coefficients on the training set.

In [ ]:
def make_logistic_data(n=700, d=18):
    X = rng.normal(size=(n, d))
    theta_true = rng.normal(size=d)
    logits = X @ theta_true / np.sqrt(d)
    probs = 1 / (1 + np.exp(-logits))
    y = rng.binomial(1, probs)
    y = 2 * y - 1
    return X, y

X, y = make_logistic_data()
X_train, y_train = X[:500], y[:500]
X_test, y_test = X[500:], y[500:]
mu = 0.03


def logistic_loss(theta):
    margins = y_train * (X_train @ theta)
    return np.mean(np.logaddexp(0, -margins)) + 0.5 * mu * theta @ theta


def logistic_grad(theta):
    margins = y_train * (X_train @ theta)
    coeff = -y_train / (1 + np.exp(np.clip(margins, -40, 40)))
    return X_train.T @ coeff / len(y_train) + mu * theta


def test_error(theta):
    pred = np.sign(X_test @ theta)
    pred[pred == 0] = 1
    return np.mean(pred != y_test)


## 2. BFGS Quasi-Newton Method

BFGS learns curvature information from gradients. Update the inverse-Hessian estimate when the curvature condition holds.

In [ ]:
def armijo_step(f, grad, x, direction, c=1e-4, shrink=0.5):
    step = 1.0
    fx = f(x)
    g = grad(x)
    while f(x + step * direction) > fx + c * step * (g @ direction):
        step *= shrink
        if step < 1e-10:
            break
    return step


def bfgs(f, grad, x0, n_iter=80):
    x = x0.copy()
    H = np.eye(len(x0))
    values, grad_norms, errors = [], [], []
    for _ in range(n_iter):
        values.append(f(x))
        g = grad(x)
        grad_norms.append(np.linalg.norm(g))
        errors.append(test_error(x))
        direction = -H @ g
        if direction @ g >= 0:
            direction = -g
            H = np.eye(len(x0))
        step = armijo_step(f, grad, x, direction)
        x_new = x + step * direction
        s = x_new - x
        y_diff = grad(x_new) - g
        sy = s @ y_diff
        if sy > 1e-10:
            rho = 1 / sy
            I = np.eye(len(x0))
            H = (I - rho * np.outer(s, y_diff)) @ H @ (I - rho * np.outer(y_diff, s)) + rho * np.outer(s, s)
        x = x_new
    return x, np.array(values), np.array(grad_norms), np.array(errors)


def gradient_descent(x0, step=0.8, n_iter=80):
    x = x0.copy()
    values, grad_norms, errors = [], [], []
    for _ in range(n_iter):
        values.append(logistic_loss(x))
        g = logistic_grad(x)
        grad_norms.append(np.linalg.norm(g))
        errors.append(test_error(x))
        x -= step * g
    return x, np.array(values), np.array(grad_norms), np.array(errors)

theta0 = np.zeros(X_train.shape[1])
_, gd_values, gd_grad, gd_err = gradient_descent(theta0)
_, bfgs_values, bfgs_grad, bfgs_err = bfgs(logistic_loss, logistic_grad, theta0)
print('final GD test error:', gd_err[-1].round(3))
print('final BFGS test error:', bfgs_err[-1].round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].semilogy(gd_values, label='GD')
axes[0].semilogy(bfgs_values, label='BFGS')
axes[0].set_title('objective')
axes[1].semilogy(gd_grad, label='GD')
axes[1].semilogy(bfgs_grad, label='BFGS')
axes[1].set_title('gradient norm')
axes[2].plot(gd_err, label='GD')
axes[2].plot(bfgs_err, label='BFGS')
axes[2].set_title('test error')
for ax in axes:
    ax.set_xlabel('iteration')
    ax.legend()
fig.tight_layout();

## 3. Projected Gradient on a Ball Constraint

Projected gradient is the clean baseline for constrained optimization: take a gradient step, then project back to the feasible set.

In [ ]:
def quad_problem(n=2):
    Q = np.array([[4.0, 1.2], [1.2, 2.0]])
    b = np.array([3.0, 1.0])
    return Q, b

Q, b = quad_problem()

def qfun(x):
    return 0.5 * x @ Q @ x - b @ x


def qgrad(x):
    return Q @ x - b


def project_ball(x, radius=0.65):
    norm = np.linalg.norm(x)
    return x if norm <= radius else radius * x / norm


def projected_gradient(x0, step=0.18, n_iter=120):
    x = x0.copy()
    path, values = [], []
    for _ in range(n_iter):
        path.append(x.copy())
        values.append(qfun(x))
        x = project_ball(x - step * qgrad(x))
    return np.array(path), np.array(values)

pg_path, pg_values = projected_gradient(np.array([-0.5, 0.5]))
print('projected solution:', pg_path[-1].round(4), 'norm:', np.linalg.norm(pg_path[-1]).round(4))
plt.plot(pg_values)
plt.xlabel('iteration')
plt.ylabel('constrained objective')
plt.title('Projected gradient objective');

## 4. Penalty Method Intuition

A penalty method solves a sequence of unconstrained problems where infeasible points become increasingly expensive.

In [ ]:
def penalty_objective(x, rho, radius=0.65):
    violation = max(0.0, np.linalg.norm(x) - radius)
    return qfun(x) + rho * violation ** 2


def numerical_grad(f, x, eps=1e-6):
    g = np.zeros_like(x)
    for i in range(len(x)):
        e = np.zeros_like(x); e[i] = eps
        g[i] = (f(x + e) - f(x - e)) / (2 * eps)
    return g

x = np.array([0.0, 0.0])
penalty_path = []
for rho in [1, 5, 20, 80]:
    for _ in range(80):
        x -= 0.04 * numerical_grad(lambda z, rho=rho: penalty_objective(z, rho), x)
    penalty_path.append((rho, x.copy(), qfun(x), np.linalg.norm(x)))
for rho, x, val, norm in penalty_path:
    print(f'rho={rho:2d} x={x.round(3)} objective={val:.3f} norm={norm:.3f}')

## Augmented Lagrangian for the ball constraint

Write the constraint as g(x) = xᵀx − r² ≤ 0. After minimizing each augmented objective, update the nonnegative multiplier by projecting λ + ρg(x) onto the positive half-line. Check feasibility, stationarity and complementarity at the final iterate.

In [ ]:
from scipy.optimize import minimize

radius, penalty = 0.65, 5.0
multiplier = 0.0
point = np.zeros(2)
for iteration in range(40):
    def objective(x):
        g = x @ x - radius**2
        shifted = max(0, multiplier + penalty*g)
        return qfun(x) + (shifted**2 - multiplier**2)/(2*penalty), qgrad(x) + 2*shifted*x
    fit = minimize(objective, point, jac=True, method="BFGS", options={"gtol": 1e-10})
    point = fit.x
    violation = point @ point - radius**2
    multiplier = max(0, multiplier + penalty*violation)
    stationarity = np.linalg.norm(qgrad(point) + 2*multiplier*point)
    if max(abs(multiplier*violation), max(0, violation), stationarity) < 1e-7:
        break
assert max(abs(multiplier*violation), max(0, violation), stationarity) < 1e-7
print("Solution:", point, "norm:", np.linalg.norm(point), "multiplier:", multiplier)